In [ ]:
# load packages

import numpy as np
import torch
from tqdm import tqdm
import diffusers
from diffusers import StableDiffusionXLPipeline
import os
import pandas as pd
import subprocess

from PIL import Image
from IPython.display import display
from transformers import AutoImageProcessor, AutoModel, pipeline, AutoTokenizer, AutoModelForCausalLM

## Exanding Captions

In [ ]:
## load expansion model

model_id = "microsoft/Phi-3-mini-4k-instruct" # LLM for expanding sentences

# Load the stable version of the model from pretrained Hugging Face Library
model = AutoModelForCausalLM.from_pretrained( # Causal LM = next token prediction
    model_id,
    device_map="auto", # Checks for GPU
    torch_dtype="auto", # Looks at model config and loads necessary weights
    trust_remote_code=False,  # Only run the Hugging Face configuration
    attn_implementation="eager" # How the model calculates Attention, prioritizing stability
)

tokenizer = AutoTokenizer.from_pretrained(model_id) # Loading the tokenizer

# Set padding to avoid sentences of different lengths
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
## expanding captions in full dataset

# load data
df = pd.read_csv('final_audiocaps.csv')

def expand_caption(text):
    # This prompting helps the model focus on visual details for image generation
    prompt = f"<|user|>\nProvide a detailed visual scene for: {text}. Mention lighting and texture. Keep it under 60 words.<|end|>\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=75,
            temperature=0.7,
            do_sample=True
        )

    # extract only the assistant's new text
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("<|assistant|>")[-1].strip()

# add the column next to the original caption
print("Expanding captions for image generation...")
tqdm.pandas()
df.insert(df.columns.get_loc('caption') + 1, 'expanded_caption', df['caption'].progress_apply(expand_caption))

# save the final result to csv file
df.to_csv('final_audiocaps_expanded.csv', index=False)
print("Process complete. File saved as final_audiocaps_expanded.csv")

In [ ]:
## filtering fully expanded dataset

# load the expanded file
df = pd.read_csv('final_audiocaps_expanded.csv')

# define the cleanup logic
# we split by the end of instruction and take everything after it
def clean_expansion(text):
    if pd.isna(text):
        return text

    separator = "Keep it under 60 words."
    if separator in text:
        # Split and take the last part, then strip whitespace
        return text.split(separator)[-1].strip()
    return text

# apply cleanup to the column
df['expanded_caption'] = df['expanded_caption'].apply(clean_expansion)

# save the cleaned version
df.to_csv('final_audiocaps_cleaned.csv', index=False)

print("Cleanup complete! The instructions have been removed.")
print("Sample of cleaned text:")
print(df['expanded_caption'].head())

## Image Generation

In [ ]:
# extracting samples in batches for image generation

df_clean = pd.read_csv("final_audiocaps_cleaned.csv") # first 100 samples
df_sample = df_clean.head(100)
df_sample.to_csv("final_audiocaps_filtered_100.csv", index=False)

df_2nd_sample = df_clean.iloc[100:200] # 100 samples
df_2nd_sample.to_csv("final_audiocaps_filtered_200.csv", index=False)

df_3rd_sample = df_clean.iloc[200:300] # 100 samples
df_3rd_sample.to_csv("final_audiocaps_filtered_300.csv", index=False)

df_4th_sample = df_clean.iloc[300:400] # 100 samples
df_4th_sample.to_csv("final_audiocaps_filtered_400.csv", index=False)

df_5th_sample = df_clean.iloc[400:500] # 100 samples
df_5th_sample.to_csv("final_audiocaps_filtered_500.csv", index=False)

df_6th_sample = df_clean.iloc[500:] # remaining samples
df_6th_sample.to_csv("final_audiocaps_filtered_600.csv", index=False)

In [ ]:
# generate large images for 100-sample subset
# capstone members generated images separately to reduce running time

# load subset and stable diffusion pipeline
subset_df = pd.read_csv("final_audiocaps_filtered_600.csv") # can be any of the filtered datasets above
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

# initialize the pipeline
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16, =
    use_safetensors=True
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

for _, row in tqdm(subset_df.iterrows(),
    total = len(subset_df)):
      caption = row["expanded_caption"]
      audio_id = str(row["audiocap_id"])
      output_dir = "sdxl_images"
      model_dir = os.path.join(output_dir, audio_id)
      if not os.path.isdir(model_dir):
        os.makedirs(model_dir, exist_ok=True)

      files = [f for f in os.listdir(model_dir)]
      prompt = 'Generate an image of: ' + caption

      if len(files) >= 1:
        continue
      else:
        print(prompt, len(files))
        with torch.no_grad():
            generator = torch.Generator(device=device).manual_seed(42)
            image = pipe(
                prompt=prompt + ", photorealistic, highly detailed, realistic photo, 8k, cinematic lighting, sharp focus",
                negative_prompt="drawing, sketch, painting, cartoon, abstract, illustration, low quality, blurry, deformed, artistic, stylized",
                generator=generator,
                num_inference_steps=40,
                guidance_scale=8.5
            ).images[0]
            image.save(os.path.join(model_dir, "0.png"))

print("Done generating images!")

In [ ]:
# preview generated images
subset_df = pd.read_csv("final_audiocaps_filtered_600.csv")
for _, row in subset_df.iterrows():
    audio_id = str(row["audiocap_id"])
    img_path = f"sdxl_images//{audio_id}/0.png"

    if os.path.exists(img_path):
        print("Audio ID:", audio_id)
        print("Caption:", row["expanded_caption"])
        img = Image.open(img_path)
        img = img.resize((150, 150))
        display(img)
        print("-" * 60)

In [ ]:
#download generated images in zip file
shutil.make_archive('generated_images', 'zip', '/content/sdxl_images')